# CareerLens AI — Phase 5: Machine Learning Model Development

In this phase, we design, train, evaluate, and serialize our **Content-Based Recommendation Engine**.

### Core Concepts:
1. **Content-Based Filtering**: Recommends occupations by matching the attributes of the job against the user's skill profile, solving the cold-start problem.
2. **TF-IDF Vectorization**: Converts raw text and skill lists into mathematical vectors, weighting unique terms heavily while penalizing ubiquitous terms.
3. **Cosine Similarity**: Computes the geometric angle $\theta$ between the user's skill vector $\vec{u}$ and every occupational vector $\vec{v}$.

## 1. Mathematical Intuition

### Term Frequency (TF):
$$\text{TF}(t, d) = \frac{\text{Count of term } t \text{ in document } d}{\text{Total terms in document } d}$$

### Inverse Document Frequency (IDF):
$$\text{IDF}(t) = \log\left(\frac{1 + N}{1 + \text{df}(t)}\right) + 1$$

### Cosine Similarity:
$$\text{Cosine Similarity}(\vec{u}, \vec{v}) = \frac{\vec{u} \cdot \vec{v}}{\|\vec{u}\| \|\vec{v}\|}$$

In [ ]:
import os
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Paths
DATA_PATH = os.path.join(os.path.dirname(os.getcwd()), "data", "processed_occupations.csv")
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} preprocessed occupations.")

## 2. Fit the TfidfVectorizer
We use `ngram_range=(1, 2)` to capture both single tokens ('python', 'sql') and multi-word phrases ('machine learning', 'data science').

In [ ]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=20000,
    sublinear_tf=True
)

tfidf_matrix = vectorizer.fit_transform(df["Combined_Features"].fillna(""))
print(f"Matrix Shape: {tfidf_matrix.shape[0]:,} occupations x {tfidf_matrix.shape[1]:,} n-gram features")

## 3. Test with Sample User Skills

In [ ]:
user_skills = "Python, SQL, Machine Learning, Data Visualization, Statistics"
user_vector = vectorizer.transform([user_skills])

# Compute cosine similarities against all 1,016 occupations
similarity_scores = cosine_similarity(user_vector, tfidf_matrix).flatten()

# Rank occupations
top_indices = similarity_scores.argsort()[::-1][:5]

print(f"Query: '{user_skills}'\n")
for rank, idx in enumerate(top_indices, 1):
    title = df.iloc[idx]['Title']
    score = similarity_scores[idx] * 100
    print(f"{rank}. {title:<45} | Similarity: {score:.2f}%")